# nb_bronze_stream_vehicle_battery

**Source :** Azure Event Hub — `vehicle-battery-live` topic  
**Sink   :** `abfss://bronze@evdatalakedev2.dfs.core.windows.net/event-stream/vehicle_battery_live/`  
**Format :** Delta (append), partitioned by `event_date`  
**Trigger:** Every 30 seconds (micro-batch streaming)

## Connector choice — Kafka endpoint (no Maven JAR needed)

Azure Event Hub exposes a **Kafka-compatible endpoint** (Standard tier+).  
Spark has a **built-in Kafka connector** — no extra Maven library required,  
so it works on Unity Catalog clusters without the artifact allowlist.

| | EventHubs connector | Kafka connector (this notebook) |
|---|---|---|
| Maven JAR | Required (`azure-eventhubs-spark`) | Not needed — built into Spark |
| Allowlist issue | Yes on UC clusters | No |
| Auth | SAS connection string | SASL/SSL with SAS key |
| Tier required | Basic+ | **Standard+** |
| Metadata columns | `enqueuedTime`, `partition`, `offset` | `timestamp`, `partition`, `offset` |

**Key Vault secret needed:** `eventhub-vehicle-battery-sasl-config`  
Value format (one line):  
```
org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="<listen-connection-string>";
```

In [ ]:
# ── Cell 1: secrets ───────────────────────────────────────────────────────────
# Store the full SASL config string in Key Vault to keep the connection string
# out of notebook code. Secret name: eventhub-vehicle-battery-sasl-config
#
# Secret value (one line, no line breaks):
#   org.apache.kafka.common.security.plain.PlainLoginModule required
#   username="$ConnectionString"
#   password="Endpoint=sb://evh-ev-intelligence-dev.servicebus.windows.net/;..."
#
EH_SASL_CONFIG = dbutils.secrets.get(scope="ev-kv-scope", key="eventhub-vehicle-battery-sasl-config")

SP_CLIENT_ID  = dbutils.secrets.get(scope="ev-kv-scope", key="sp-client-id")
SP_CLIENT_SEC = dbutils.secrets.get(scope="ev-kv-scope", key="sp-client-secret")
SP_TENANT_ID  = dbutils.secrets.get(scope="ev-kv-scope", key="sp-tenant-id")
ADLS_ACCOUNT  = dbutils.secrets.get(scope="ev-kv-scope", key="adls-account-name")

print("Secrets loaded.")

In [ ]:
# ── Cell 2: ADLS OAuth config ─────────────────────────────────────────────────
spark.conf.set(f"fs.azure.account.auth.type.{ADLS_ACCOUNT}.dfs.core.windows.net",       "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{ADLS_ACCOUNT}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{ADLS_ACCOUNT}.dfs.core.windows.net",     SP_CLIENT_ID)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{ADLS_ACCOUNT}.dfs.core.windows.net", SP_CLIENT_SEC)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{ADLS_ACCOUNT}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token")
print("ADLS OAuth configured.")

In [ ]:
# ── Cell 3: paths + Kafka/Event Hub config ────────────────────────────────────
BRONZE_PATH     = f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/event-stream/vehicle_battery_live/"
CHECKPOINT_PATH = f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/_checkpoints/vehicle-battery-live/"

EH_NAMESPACE  = "evh-ev-intelligence-dev"          # your Event Hub namespace name
EH_TOPIC      = "vehicle-battery-live"              # topic name
EH_BOOTSTRAP  = f"{EH_NAMESPACE}.servicebus.windows.net:9093"  # Kafka endpoint

KAFKA_OPTIONS = {
    "kafka.bootstrap.servers":                    EH_BOOTSTRAP,
    "subscribe":                                  EH_TOPIC,
    "kafka.security.protocol":                    "SASL_SSL",
    "kafka.sasl.mechanism":                       "PLAIN",
    "kafka.sasl.jaas.config":                     EH_SASL_CONFIG,
    "startingOffsets":                            "latest",   # latest = only new events
    "maxOffsetsPerTrigger":                       1000,        # cap events per batch
    "failOnDataLoss":                             "false",     # handle Event Hub retention gaps
}

print(f"Bootstrap server : {EH_BOOTSTRAP}")
print(f"Topic            : {EH_TOPIC}")
print(f"Bronze path      : {BRONZE_PATH}")
print(f"Checkpoint       : {CHECKPOINT_PATH}")

In [ ]:
# ── Cell 4: read stream via Kafka connector ───────────────────────────────────
# Kafka delivers: key (binary), value (binary), topic, partition, offset,
#                 timestamp (enqueue time), timestampType, headers
raw_stream = (
    spark.readStream
    .format("kafka")
    .options(**KAFKA_OPTIONS)
    .load()
)

print("Raw Kafka stream schema:")
raw_stream.printSchema()

In [ ]:
# ── Cell 5: parse JSON from the Kafka value column ────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

EVENT_SCHEMA = StructType([
    StructField("event_id",                   StringType(),  True),
    StructField("vehicle_id",                 StringType(),  True),
    StructField("session_id",                 StringType(),  True),
    StructField("station_id",                 StringType(),  True),
    StructField("charger_id",                 StringType(),  True),
    StructField("battery_pct",                DoubleType(),  True),
    StructField("charging_rate_kw",           DoubleType(),  True),
    StructField("battery_temp_c",             DoubleType(),  True),
    StructField("state_of_charge_target_pct", IntegerType(), True),
    StructField("estimated_minutes_to_full",  IntegerType(), True),
    StructField("event_ts",                   StringType(),  True),
])

parsed = (
    raw_stream
    # Kafka value is binary — cast to string first
    .withColumn("value_str", F.col("value").cast("string"))
    .withColumn("parsed",    F.from_json(F.col("value_str"), EVENT_SCHEMA))
    .select(
        F.col("parsed.*"),
        # Kafka metadata columns
        F.col("timestamp") .alias("eh_enqueued_time"),   # Event Hub enqueue time
        F.col("partition") .alias("eh_partition"),
        F.col("offset")    .alias("eh_offset"),
        F.col("topic")     .alias("eh_topic"),
    )
)

print("Parsed schema:")
parsed.printSchema()

In [ ]:
# ── Cell 6: add Bronze metadata columns ──────────────────────────────────────
bronze_df = (
    parsed
    .withColumn("_ingestion_ts", F.current_timestamp())
    .withColumn("_source",       F.lit("eventhub/vehicle-battery-live"))
    # Flag rows where JSON parsing failed (value was not valid JSON)
    .withColumn("_is_corrupt",   F.col("event_id").isNull())
    # Partition column for Hive-style partitioning on ADLS
    .withColumn("event_date",    F.to_date(F.col("event_ts")))
)

print("Bronze schema:")
bronze_df.printSchema()

In [ ]:
# ── Cell 7: write stream to Bronze Delta ─────────────────────────────────────
streaming_query = (
    bronze_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .partitionBy("event_date")
    .trigger(processingTime="30 seconds")
    .start(BRONZE_PATH)
)

print(f"Stream started. Query ID: {streaming_query.id}")

In [ ]:
# ── Cell 8: keep-alive monitor loop ──────────────────────────────────────────
import time

while streaming_query.isActive:
    prog = streaming_query.lastProgress
    if prog:
        rows_in  = prog.get("numInputRows", 0)
        rows_sec = prog.get("inputRowsPerSecond", 0.0)
        batch    = prog.get("batchId", "?")
        print(f"[Batch {batch}] inputRows={rows_in}  rate={rows_sec:.1f} rows/s")
    time.sleep(30)

streaming_query.awaitTermination()

In [ ]:
# ── Cell 9: optional verify (run manually, not in job) ───────────────────────
verify_df = spark.read.format("delta").load(BRONZE_PATH)
print(f"Total Bronze rows: {verify_df.count()}")
verify_df.orderBy(F.col("_ingestion_ts").desc()).limit(5).display()